# Llama 3.1 Cybersecurity Fine-Tuning (QLoRA)

This notebook fine-tunes a Llama 3.1 model for Apocalypse data and now supports both:
- instruction datasets that already contain `text`
- atomic ATT&CK datasets (`input.signals` + `output`) converted to `text` on the fly

Workflow:
1. Install dependencies
2. Configure dataset mode (`atomic` or `instruction`)
3. Load JSONL files and auto-build `text` samples
4. Configure 4-bit QLoRA
5. Train with TRL SFTTrainer
6. Save adapter weights and run a quick sanity generation

## Prerequisites

- Preferred: NVIDIA GPU runtime (local or Colab).
- Hugging Face access to the selected checkpoint.
- This notebook supports two dataset styles:
  - `instruction` (rows already contain `text`)
  - `atomic` (rows contain `input.signals` + `output.{tactic,technique}` and notebook will build `text`)

### Runtime Options

- Local laptop runtime: works if you have CUDA + enough VRAM.
- Colab runtime (via Colab extension): recommended if local GPU is limited.
- CPU-only run is supported for smoke validation when `RUN_MODE=smoke`.

### Run Modes

- `RUN_MODE=full` (default): real QLoRA training mode using `meta-llama/Llama-3.1-8B-Instruct` (requires GPU + HF token).
- `RUN_MODE=smoke`: CPU-safe validation mode using a tiny public model and small sample sizes.
- In `RUN_MODE=full`, this notebook uses only your provided dataset files (no automatic data download/build).

### Hugging Face Token

- Put `HUGGINGFACE_ACCESS_TOKEN=...` in a `.env` that exists inside the active Colab runtime project folder.
- Or set it directly as an environment variable in the notebook runtime.
- In `RUN_MODE=full`, token is required and notebook will fail fast if missing.

### Custom Dataset Paths (Google Drive Friendly)

- You can provide dataset files directly with env variables:
  - `TRAIN_FILE_PATH=/content/drive/MyDrive/.../train.jsonl`
  - `EVAL_FILE_PATH=/content/drive/MyDrive/.../eval.jsonl` (optional)
- Optional: set `DATA_ROOT=/content/drive/MyDrive/.../your_dataset_folder` to use default filenames from that folder.
- If eval file is missing, notebook can split train automatically.

In [1]:
%pip install -q "transformers>=4.44.0" "datasets>=2.21.0" "accelerate>=0.33.0" "peft>=0.12.0" "trl>=0.10.1" "bitsandbytes>=0.43.2" "huggingface_hub>=0.24.0" "python-dotenv>=1.0.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 19.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.7 MB/s eta 0:00:00:00:0100:01


In [1]:
from pathlib import Path
import json
import os

import torch
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    )
from trl import SFTTrainer

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def _dedupe_paths(paths):
    seen = set()
    output = []
    for path in paths:
        try:
            resolved = path.expanduser().resolve()
        except Exception:
            resolved = path
        key = str(resolved)
        if key not in seen:
            seen.add(key)
            output.append(resolved)
    return output


def _candidate_roots(cwd: Path):
    env_roots = []
    for var_name in ("PROJECT_ROOT", "APOCALYPSE_TRAINING_ROOT", "COLAB_PROJECT_ROOT", "GITHUB_WORKSPACE"):
        value = os.getenv(var_name)
        if value:
            env_roots.append(Path(value))

    common_roots = [
        Path("/content/Apocalypse-Training"),
        Path("/content/drive/MyDrive/Apocalypse-Training"),
        Path("/content/drive/MyDrive/Colab Notebooks/Apocalypse-Training"),
        Path("/workspace/Apocalypse-Training"),
        Path("/workspaces/Apocalypse-Training"),
    ]

    return _dedupe_paths(env_roots + [cwd] + list(cwd.parents) + common_roots)


def _project_score(candidate: Path) -> int:
    score = 0
    candidate_str = str(candidate).lower()
    if "apocalypse-training" in candidate_str:
        score += 5
    if (candidate / "apocalypse-context.md").exists():
        score += 8
    if (candidate / "notebooks/train_llama31_qlora.ipynb").exists():
        score += 6
    if (candidate / "data/raw/atomic/atomic_enterprise-attack_with_subtechniques.jsonl").exists():
        score += 10
    if (candidate / "data").exists():
        score += 2
    if (candidate / "notebooks").exists():
        score += 2
    return score


def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = _candidate_roots(cwd)
    scored = sorted(((_project_score(candidate), candidate) for candidate in candidates), reverse=True)
    if scored and scored[0][0] > 0:
        return scored[0][1]
    return cwd


def _resolve_existing_path(preferred_path: Path, relative_path: Path, project_root: Path, cwd: Path):
    candidates = [preferred_path]
    for root in _candidate_roots(cwd):
        candidates.append(root / relative_path)
    candidates = _dedupe_paths(candidates)

    for candidate in candidates:
        if candidate.exists():
            return candidate, [str(path) for path in candidates]

    target_name = relative_path.name
    recursive_bases = _dedupe_paths([cwd, project_root, Path("/content"), Path("/workspace"), Path("/workspaces")])
    for base in recursive_bases:
        if not base.exists():
            continue
        try:
            for match in base.rglob(target_name):
                if match.is_file():
                    return match.resolve(), [str(path) for path in candidates]
        except Exception:
            continue

    return preferred_path, [str(path) for path in candidates]


def _resolve_user_path(raw_value: str, cwd: Path):
    value = (raw_value or "").strip()
    if not value:
        return None
    path = Path(value).expanduser()
    try:
        if path.is_absolute():
            return path.resolve()
        return (cwd / path).resolve()
    except Exception:
        return path


def _pick_jsonl_files(data_root: Path):
    files = []
    if data_root.exists() and data_root.is_dir():
        try:
            files = sorted(
                [path.resolve() for path in data_root.glob("*.jsonl") if path.is_file()],
                key=lambda path: path.name.lower(),
            )
        except Exception:
            files = []

    discovered = [str(path) for path in files]
    if not files:
        return None, None, discovered

    def _first_match(paths, keywords):
        for path in paths:
            name = path.name.lower()
            if any(keyword in name for keyword in keywords):
                return path
        return None

    train_keywords = ("train", "training", "with_subtechniques", "atomic", "cases")
    eval_keywords = ("eval", "validation", "valid", "val", "test", "balanced")

    train_file = _first_match(files, train_keywords) or files[0]
    remaining = [path for path in files if path != train_file]
    eval_file = _first_match(remaining, eval_keywords)

    if eval_file is None and remaining and any(tag in train_file.name.lower() for tag in ("train", "training")):
        eval_file = remaining[0]

    return train_file, eval_file, discovered


cwd = Path.cwd().resolve()
ROOT_CANDIDATES = [str(path) for path in _candidate_roots(cwd)]
PROJECT_ROOT = _find_project_root()

SEARCHED_ENV_PATHS = []
LOADED_ENV_FILES = []
for root in _dedupe_paths([PROJECT_ROOT, cwd] + list(cwd.parents)):
    env_path = root / ".env"
    SEARCHED_ENV_PATHS.append(str(env_path))
    if env_path.exists():
        load_dotenv(env_path, override=False)
        LOADED_ENV_FILES.append(str(env_path))

HF_TOKEN = os.getenv("HUGGINGFACE_ACCESS_TOKEN") or os.getenv("HF_TOKEN")
HF_TOKEN_SOURCE = "env" if HF_TOKEN else "not-found"
SECRET_ATTEMPTS = []

if IN_COLAB and not HF_TOKEN:
    try:
        from google.colab import userdata  # type: ignore
        secret_candidates = []
        explicit_secret_name = os.getenv("COLAB_HF_SECRET_NAME", "").strip()
        explicit_secret_names = os.getenv("COLAB_HF_SECRET_NAMES", "").strip()
        if explicit_secret_name:
            secret_candidates.append(explicit_secret_name)
        if explicit_secret_names:
            secret_candidates.extend([name.strip() for name in explicit_secret_names.split(",") if name.strip()])
        secret_candidates.extend(["HUGGINGFACE_ACCESS_TOKEN", "HF_TOKEN", "secretName"])
        seen_secret_names = set()
        for secret_name in secret_candidates:
            if not secret_name or secret_name in seen_secret_names:
                continue
            seen_secret_names.add(secret_name)
            try:
                secret_value = userdata.get(secret_name)
            except Exception as secret_error:
                SECRET_ATTEMPTS.append(f"{secret_name}:{type(secret_error).__name__}")
                secret_value = None
            if secret_value and str(secret_value).strip():
                HF_TOKEN = str(secret_value).strip()
                os.environ["HUGGINGFACE_ACCESS_TOKEN"] = HF_TOKEN
                HF_TOKEN_SOURCE = f"colab_userdata:{secret_name}"
                SECRET_ATTEMPTS.append(f"{secret_name}:ok")
                break
            if secret_value is None:
                continue
            SECRET_ATTEMPTS.append(f"{secret_name}:empty")
    except Exception as error:
        SECRET_ATTEMPTS.append(f"userdata_access:{type(error).__name__}")

RUN_MODE = os.getenv("RUN_MODE", "full").strip().lower()
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")
SMOKE_TEST_MODE = RUN_MODE == "smoke"

FULL_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
SMOKE_MODEL_NAME = "sshleifer/tiny-gpt2"
MODEL_NAME = SMOKE_MODEL_NAME if SMOKE_TEST_MODE else FULL_MODEL_NAME

if not SMOKE_TEST_MODE and MODEL_NAME != FULL_MODEL_NAME:
    raise RuntimeError("Full mode must use meta-llama/Llama-3.1-8B-Instruct")

TOKEN_REQUIRED = not SMOKE_TEST_MODE

if HF_TOKEN:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as error:
        print(f"Warning: Hugging Face login failed ({error}).")

# Switch here: "atomic" (direct ATT&CK JSONL) or "instruction" (already formatted).
DATASET_MODE = "atomic"

if DATASET_MODE == "atomic":
    train_relative = Path("data/raw/atomic/atomic_enterprise-attack_with_subtechniques.jsonl")
    eval_relative = Path("data/raw/atomic/atomic_enterprise-attack_with_subtechniques_balanced.jsonl")
else:
    train_relative = Path("data/training/train.instructions.jsonl")
    eval_relative = Path("data/evaluation/eval.sample.jsonl")

TRAIN_FILE_OVERRIDE = os.getenv("TRAIN_FILE_PATH", "").strip() or os.getenv("TRAIN_JSONL_PATH", "").strip()
EVAL_FILE_OVERRIDE = os.getenv("EVAL_FILE_PATH", "").strip() or os.getenv("EVAL_JSONL_PATH", "").strip()
DATA_ROOT_OVERRIDE = os.getenv("DATA_ROOT", "").strip() or os.getenv("DATA_DIR", "").strip()

AUTO_ATOMIC_SOURCE = "none"
AUTO_ATOMIC_DISCOVERED = []

if DATASET_MODE == "atomic" and IN_COLAB and (not TRAIN_FILE_OVERRIDE) and (not DATA_ROOT_OVERRIDE):
    default_atomic_dir = Path("/content/atomic")
    auto_train, auto_eval, auto_discovered = _pick_jsonl_files(default_atomic_dir)
    AUTO_ATOMIC_DISCOVERED = auto_discovered
    if auto_train is not None:
        TRAIN_FILE_OVERRIDE = str(auto_train)
        if auto_eval is not None:
            EVAL_FILE_OVERRIDE = str(auto_eval)
        DATA_ROOT_OVERRIDE = str(default_atomic_dir)
        AUTO_ATOMIC_SOURCE = str(default_atomic_dir)

if DATA_ROOT_OVERRIDE:
    data_root_path = _resolve_user_path(DATA_ROOT_OVERRIDE, cwd)
    if data_root_path is not None:
        if DATASET_MODE == "atomic" and not TRAIN_FILE_OVERRIDE:
            root_train, root_eval, root_discovered = _pick_jsonl_files(data_root_path)
            if root_train is not None:
                TRAIN_FILE_OVERRIDE = str(root_train)
                if (not EVAL_FILE_OVERRIDE) and (root_eval is not None):
                    EVAL_FILE_OVERRIDE = str(root_eval)
                if AUTO_ATOMIC_SOURCE == "none":
                    AUTO_ATOMIC_DISCOVERED = root_discovered
            else:
                TRAIN_FILE_OVERRIDE = str(data_root_path / train_relative.name)
                if not EVAL_FILE_OVERRIDE:
                    EVAL_FILE_OVERRIDE = str(data_root_path / eval_relative.name)
        elif not TRAIN_FILE_OVERRIDE:
            TRAIN_FILE_OVERRIDE = str(data_root_path / train_relative.name)
            if not EVAL_FILE_OVERRIDE:
                EVAL_FILE_OVERRIDE = str(data_root_path / eval_relative.name)

resolved_train_override = _resolve_user_path(TRAIN_FILE_OVERRIDE, cwd) if TRAIN_FILE_OVERRIDE else None
resolved_eval_override = _resolve_user_path(EVAL_FILE_OVERRIDE, cwd) if EVAL_FILE_OVERRIDE else None

if resolved_train_override is not None:
    TRAIN_FILE = resolved_train_override
    SEARCHED_TRAIN_PATHS = [str(TRAIN_FILE)]
else:
    TRAIN_FILE, SEARCHED_TRAIN_PATHS = _resolve_existing_path(
        PROJECT_ROOT / train_relative,
        train_relative,
        PROJECT_ROOT,
        cwd,
        )

if resolved_eval_override is not None:
    EVAL_FILE = resolved_eval_override
    SEARCHED_EVAL_PATHS = [str(EVAL_FILE)]
else:
    EVAL_FILE, SEARCHED_EVAL_PATHS = _resolve_existing_path(
        PROJECT_ROOT / eval_relative,
        eval_relative,
        PROJECT_ROOT,
        cwd,
        )

AUTO_SPLIT_EVAL_IF_MISSING = True
EVAL_SPLIT_RATIO = 0.05

if SMOKE_TEST_MODE:
    MAX_TRAIN_SAMPLES = 32
    MAX_EVAL_SAMPLES = 8
    MAX_SEQ_LENGTH = 256
    NUM_EPOCHS = 1
    LEARNING_RATE = 5e-5
    BATCH_SIZE = 1
    GRAD_ACCUM_STEPS = 1
else:
    MAX_TRAIN_SAMPLES = 0
    MAX_EVAL_SAMPLES = 0
    MAX_SEQ_LENGTH = 1024
    NUM_EPOCHS = 1
    LEARNING_RATE = 2e-5
    BATCH_SIZE = 1
    GRAD_ACCUM_STEPS = 8

SEED = 42
OUTPUT_DIR = PROJECT_ROOT / "artifacts/llama31-cyber-qlora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"In Colab: {IN_COLAB}")
print(f"CWD: {cwd}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Root candidates scanned: {len(ROOT_CANDIDATES)}")
print(f"Searched .env paths: {len(SEARCHED_ENV_PATHS)}")
print(f"Loaded .env files: {LOADED_ENV_FILES if LOADED_ENV_FILES else 'none'}")
print(f"Run mode: {RUN_MODE}")
print(f"Model name: {MODEL_NAME}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Train file override: {str(resolved_train_override) if resolved_train_override else 'none'}")
print(f"Eval file override: {str(resolved_eval_override) if resolved_eval_override else 'none'}")
print(f"Data root override: {DATA_ROOT_OVERRIDE if DATA_ROOT_OVERRIDE else 'none'}")
print(f"Auto atomic source: {AUTO_ATOMIC_SOURCE}")
if AUTO_ATOMIC_DISCOVERED:
    print(f"Auto atomic discovered files: {AUTO_ATOMIC_DISCOVERED[:10]}")
print(f"Train file: {TRAIN_FILE}")
print(f"Eval file: {EVAL_FILE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"HF token detected: {bool(HF_TOKEN)}")
print(f"HF token source: {HF_TOKEN_SOURCE}")
if SECRET_ATTEMPTS:
    print(f"Colab secret attempts: {SECRET_ATTEMPTS}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif SMOKE_TEST_MODE:
    print("CPU smoke test mode enabled: this validates notebook flow before full GPU training.")
else:
    print("Warning: CUDA not detected. Full QLoRA training requires Colab GPU or local NVIDIA GPU.")

if TOKEN_REQUIRED and not HF_TOKEN:
    raise RuntimeError(
        "HUGGINGFACE_ACCESS_TOKEN is required in RUN_MODE=full for gated model access. "
        "If you use Colab secrets, add one of: HUGGINGFACE_ACCESS_TOKEN, HF_TOKEN, secretName; "
        "or set COLAB_HF_SECRET_NAME to your secret key name, then rerun Cell 4."
    )

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In Colab: True
CWD: /content
Project root: /workspaces/Apocalypse-Training
Root candidates scanned: 7
Searched .env paths: 3
Loaded .env files: ['/content/.env']
Run mode: full
Model name: meta-llama/Llama-3.1-8B-Instruct
Dataset mode: atomic
Train file override: /content/atomic/atomic_enterprise-attack.jsonl
Eval file override: /content/atomic/atomic_enterprise-attack_with_subtechniques_balanced.jsonl
Data root override: /content/atomic
Auto atomic source: /content/atomic
Auto atomic discovered files: ['/content/atomic/atomic_enterprise-attack.jsonl', '/content/atomic/atomic_enterprise-attack_with_subtechniques.jsonl', '/content/atomic/atomic_enterprise-attack_with_subtechniques_balanced.jsonl', '/content/atomic/atomic_enterprise-attack_with_subtechniques_plus_recon.jsonl']
Train file: /content/atomic/atomic_enterprise-attack.jsonl
Eval file: /content/atomic/atomic_enterprise-attack_with_subtechniques_balanced.jsonl
Output dir: /workspaces/Apocalypse-Training/artifacts/llama31-cyber-q

In [2]:
# Run override for current runtime (/content):
# - Train on merged reconnaissance file via TRAIN_FILE_PATH
# - Keep eval fixed
# - Train for 3 epochs
atomic_root = Path("/content/atomic")
BASE_TRAIN_FILE = atomic_root / "atomic_enterprise-attack_with_subtechniques.jsonl"
TRAIN_FILE = atomic_root / "atomic_enterprise-attack_with_subtechniques_plus_recon.jsonl"
EVAL_FILE = atomic_root / "atomic_enterprise-attack_with_subtechniques_balanced.jsonl"

def _tools_for_recon(technique_name: str):
    name = technique_name.lower()
    if "wordlist" in name:
        return ["ffuf", "gobuster", "dirsearch"]
    if "vulnerability" in name:
        return ["nuclei", "nessus", "openvas"]
    if "scanning" in name:
        return ["nmap", "masscan", "zmap"]
    if "dns" in name or "whois" in name:
        return ["dig", "whois", "dnsrecon"]
    if "certificate" in name or "cdn" in name:
        return ["crtsh", "openssl", "securitytrails"]
    if "social" in name:
        return ["linkedin", "x-search", "osintgram"]
    if "repository" in name or "code" in name:
        return ["github", "gitlab", "trufflehog"]
    if "phishing" in name or "spearphishing" in name:
        return ["mail-templates", "voip", "tracking-links"]
    if "identity" in name or "employee" in name or "email" in name:
        return ["theHarvester", "hunter", "maltego"]
    return ["shodan", "censys", "amass"]

assert BASE_TRAIN_FILE.exists(), f"Missing base train file: {BASE_TRAIN_FILE}"
assert EVAL_FILE.exists(), f"Missing fixed eval file: {EVAL_FILE}"

if not TRAIN_FILE.exists():
    base_rows = []
    with BASE_TRAIN_FILE.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                base_rows.append(json.loads(line))

    recon_rows = [row for row in base_rows if row.get("output", {}).get("tactic") == "reconnaissance"]
    augmented_rows = []
    for index, row in enumerate(recon_rows, start=1):
        clone = json.loads(json.dumps(row, ensure_ascii=True))
        base_id = str(clone.get("id", f"atomic-recon-{index:03d}"))
        clone["id"] = f"{base_id}-apoc-aug-{index:03d}"
        technique = str(clone.get("output", {}).get("technique", "")).strip()
        tools = _tools_for_recon(technique)
        signals = clone.setdefault("input", {}).setdefault("signals", [])
        if isinstance(signals, list):
            signals.append(
                f"Tooling context: reconnaissance can involve {', '.join(tools)} depending on target surface and objective."
            )
        meta = clone.setdefault("meta", {})
        meta["source"] = "apocalypse_recon_augmented_in_kernel"
        meta["tools"] = tools
        augmented_rows.append(clone)

    with TRAIN_FILE.open("w", encoding="utf-8") as handle:
        for row in base_rows:
            handle.write(json.dumps(row, ensure_ascii=True) + "\n")
        for row in augmented_rows:
            handle.write(json.dumps(row, ensure_ascii=True) + "\n")

    print(f"Created merged train file: {TRAIN_FILE}")
    print(f"- Base rows: {len(base_rows)}")
    print(f"- Added recon rows: {len(augmented_rows)}")

TRAIN_FILE_OVERRIDE = str(TRAIN_FILE)
EVAL_FILE_OVERRIDE = str(EVAL_FILE)
SEARCHED_TRAIN_PATHS = [str(TRAIN_FILE)]
SEARCHED_EVAL_PATHS = [str(EVAL_FILE)]

AUTO_SPLIT_EVAL_IF_MISSING = False
NUM_EPOCHS = 3

print("Run override active:")
print(f"- Train file (TRAIN_FILE_PATH): {TRAIN_FILE_OVERRIDE}")
print(f"- Eval file (fixed): {EVAL_FILE_OVERRIDE}")
print(f"- Epochs: {NUM_EPOCHS}")
print(f"- Eval fixed: {not AUTO_SPLIT_EVAL_IF_MISSING}")

Run override active:
- Train file (TRAIN_FILE_PATH): /content/atomic/atomic_enterprise-attack_with_subtechniques_plus_recon.jsonl
- Eval file (fixed): /content/atomic/atomic_enterprise-attack_with_subtechniques_balanced.jsonl
- Epochs: 3
- Eval fixed: True


In [3]:
def _write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=True) + "\n")

def _validate_atomic_rows(ds, split_name: str, max_checks: int = 256):
    if len(ds) == 0:
        raise ValueError(f"{split_name} dataset is empty.")

    checks = min(max_checks, len(ds))
    for index in range(checks):
        row = ds[index]
        missing = []

        input_obj = row.get("input")
        if not isinstance(input_obj, dict):
            missing.append("input")
        else:
            signals = input_obj.get("signals")
            if not isinstance(signals, (list, str)):
                missing.append("input.signals")

        output_obj = row.get("output")
        if not isinstance(output_obj, dict):
            missing.append("output")
        else:
            if "tactic" not in output_obj:
                missing.append("output.tactic")
            if "technique" not in output_obj:
                missing.append("output.technique")

        if missing:
            raise ValueError(
                f"{split_name} row {index} is missing required atomic fields: {missing}. "
                "Each row must include input.signals and output.tactic/output.technique."
            )

if (not TRAIN_FILE.exists()) and SMOKE_TEST_MODE:
    print("Train dataset not found in smoke mode; generating a tiny synthetic atomic dataset for validation.")
    fallback_dir = PROJECT_ROOT / "artifacts/smoke_data"
    fallback_train = fallback_dir / "atomic_smoke_train.jsonl"
    fallback_eval = fallback_dir / "atomic_smoke_eval.jsonl"

    smoke_examples = [
        ("T1059", "execution", "Command and Scripting Interpreter"),
        ("T1078", "defense_evasion", "Valid Accounts"),
        ("T1110", "credential_access", "Brute Force"),
        ("T1083", "discovery", "File and Directory Discovery"),
        ("T1041", "exfiltration", "Exfiltration Over C2 Channel"),
        ("T1105", "command_and_control", "Ingress Tool Transfer"),
        ("T1021", "lateral_movement", "Remote Services"),
        ("T1036", "defense_evasion", "Masquerading"),
        ("T1071", "command_and_control", "Application Layer Protocol"),
        ("T1003", "credential_access", "OS Credential Dumping"),
    ]

    smoke_rows = []
    for index, (attack_id, tactic, technique) in enumerate(smoke_examples, start=1):
        smoke_rows.append(
            {
                "id": f"smoke-{index:03d}",
                "input": {
                    "signals": [
                        f"Observed behavior linked to {technique} ({attack_id}).",
                        f"Threat intel context: simulated example for notebook smoke validation ({tactic}).",
                    ]
                },
                "output": {"tactic": tactic, "technique": technique},
                "meta": {
                    "source": "smoke_synthetic",
                    "difficulty": "easy",
                    "attack_stage": tactic,
                    "attack_id": attack_id,
                    "is_subtechnique": False,
                },
            }
        )

    split_index = max(1, int(len(smoke_rows) * 0.8))
    _write_jsonl(fallback_train, smoke_rows[:split_index])
    _write_jsonl(fallback_eval, smoke_rows[split_index:])

    TRAIN_FILE = fallback_train
    EVAL_FILE = fallback_eval
    print(f"Smoke train file: {TRAIN_FILE}")
    print(f"Smoke eval file: {EVAL_FILE}")

if IN_COLAB and str(TRAIN_FILE).startswith("/content/drive") and not Path("/content/drive").exists():
    print("Google Drive path detected for train file, but /content/drive is not mounted.")
    print("Mount Drive first, then rerun from Cell 4.")

assert TRAIN_FILE.exists(), (
    f"Missing training file: {TRAIN_FILE}\n"
    f"Searched train candidates (first 10): {SEARCHED_TRAIN_PATHS[:10]}\n"
    "For atomic mode, upload at least one JSONL file to /content/atomic or set TRAIN_FILE_PATH explicitly."
    )

if EVAL_FILE.exists():
    datasets = load_dataset(
        "json",
        data_files={"train": str(TRAIN_FILE), "eval": str(EVAL_FILE)},
    )
    train_ds = datasets["train"]
    eval_ds = datasets["eval"]
else:
    if EVAL_FILE_OVERRIDE:
        print(f"Eval file override not found: {EVAL_FILE}. Falling back to train split.")
    print(
        "Eval file not found, using automatic split from train set. "
        f"Searched eval candidates (first 10): {SEARCHED_EVAL_PATHS[:10]}"
    )
    datasets = load_dataset("json", data_files={"train": str(TRAIN_FILE)})
    train_ds = datasets["train"]
    if AUTO_SPLIT_EVAL_IF_MISSING:
        split = train_ds.train_test_split(test_size=EVAL_SPLIT_RATIO, seed=SEED, shuffle=True)
        train_ds = split["train"]
        eval_ds = split["test"]
    else:
        eval_size = min(128, len(train_ds))
        eval_ds = train_ds.select(range(eval_size))

if DATASET_MODE == "atomic":
    _validate_atomic_rows(train_ds, "train")
    _validate_atomic_rows(eval_ds, "eval")

def _render_text(instruction: str, input_payload: dict, expected_output: dict) -> str:
    return (
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{json.dumps(input_payload, ensure_ascii=True)}\n\n"
        "### Expected Output:\n"
        f"{json.dumps(expected_output, ensure_ascii=True)}"
    )

def _atomic_to_text(example: dict) -> str:
    signals = example.get("input", {}).get("signals", [])
    if not isinstance(signals, list):
        signals = [signals]
    input_payload = {"signals": signals}
    expected_output = {
        "tactic": example.get("output", {}).get("tactic", ""),
        "technique": example.get("output", {}).get("technique", ""),
    }
    instruction = "Identify the most likely MITRE ATT&CK tactic and primary technique from observed behavior signals."
    return _render_text(instruction, input_payload, expected_output)

def _instruction_to_text(example: dict) -> str:
    existing = example.get("text")
    if isinstance(existing, str) and existing.strip():
        return existing
    instruction = str(example.get("instruction", "Analyze the cybersecurity scenario."))
    input_payload = example.get("input", {})
    expected_output = example.get("expected_output", {})
    return _render_text(instruction, input_payload, expected_output)

def _ensure_text(example: dict) -> dict:
    if DATASET_MODE == "atomic":
        return {"text": _atomic_to_text(example)}
    return {"text": _instruction_to_text(example)}

train_ds = train_ds.map(_ensure_text)
eval_ds = eval_ds.map(_ensure_text)

if MAX_TRAIN_SAMPLES > 0:
    train_ds = train_ds.select(range(min(MAX_TRAIN_SAMPLES, len(train_ds))))
if MAX_EVAL_SAMPLES > 0:
    eval_ds = eval_ds.select(range(min(MAX_EVAL_SAMPLES, len(eval_ds))))

required_column = "text"
assert required_column in train_ds.column_names, f"Expected 'text' in train columns: {train_ds.column_names}"
assert required_column in eval_ds.column_names, f"Expected 'text' in eval columns: {eval_ds.column_names}"

print(train_ds)
print(eval_ds)
print("\nSample training row (truncated):\n")
print(train_ds[0]["text"][:700])

Dataset({
    features: ['id', 'input', 'output', 'meta', 'text'],
    num_rows: 736
})
Dataset({
    features: ['id', 'input', 'output', 'meta', 'text'],
    num_rows: 210
})

Sample training row (truncated):

### Instruction:
Identify the most likely MITRE ATT&CK tactic and primary technique from observed behavior signals.

### Input:
{"signals": ["Observed behavior linked to Data Obfuscation (T1001).", "Threat intel context: Adversaries may obfuscate command and control traffic to make it more difficult to detect.(Citation: Bitdefender FunnyDream Campaign November 2020) Command and control (C2) communications are hidden (but not necessarily encrypted) in an attempt to make the content more difficult to discover or decipher and to make the communication less conspicuous and hide commands from being seen. This encompasses many methods, such as adding junk data to protocol traffic, using steganogra


In [4]:
# GPU cleanup before re-loading model for the new run.
import gc
import os

# Helps reduce allocator fragmentation for large model reloads.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for name in ["trainer", "model", "train_result", "inputs", "outputs", "generated"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    free_gb = torch.cuda.mem_get_info()[0] / (1024 ** 3)
    total_gb = torch.cuda.mem_get_info()[1] / (1024 ** 3)
    print(f"CUDA memory after cleanup: {free_gb:.2f} GiB free / {total_gb:.2f} GiB total")
else:
    print("CUDA not available; skipped GPU cache cleanup.")

CUDA memory after cleanup: 21.84 GiB free / 22.03 GiB total


In [5]:
if (not torch.cuda.is_available()) and (not SMOKE_TEST_MODE):
    raise RuntimeError(
        "CUDA GPU is required for full training mode. "
        "Set RUN_MODE='smoke' to validate notebook flow on CPU first."
    )

USE_QLORA = torch.cuda.is_available() and (not SMOKE_TEST_MODE)
auth_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {"token": False}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, **auth_kwargs)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if USE_QLORA:
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        **auth_kwargs,
    )
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    print("QLoRA mode enabled (GPU + 4-bit).")
else:
    compute_dtype = torch.float32
    bnb_config = None
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        **auth_kwargs,
    )
    peft_config = None
    if torch.cuda.is_available():
        print("Smoke mode enabled: loading non-quantized model for fast validation.")
    else:
        print("CPU smoke mode enabled: loading non-quantized model for flow validation.")

model.config.use_cache = False

print("Model and tokenizer are ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

QLoRA mode enabled (GPU + 4-bit).
Model and tokenizer are ready.


In [6]:
import inspect

training_args_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())

effective_batch_size = max(1, BATCH_SIZE * GRAD_ACCUM_STEPS)
steps_per_epoch = max(1, (len(train_ds) + effective_batch_size - 1) // effective_batch_size)
total_steps = max(1, steps_per_epoch * NUM_EPOCHS)
warmup_steps = max(0, int(0.03 * total_steps))

training_args_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "num_train_epochs": NUM_EPOCHS,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "lr_scheduler_type": "cosine",
    "eval_steps": 20,
    "save_strategy": "steps",
    "save_steps": 20,
    "logging_steps": 5,
    "max_grad_norm": 0.3,
    "weight_decay": 0.01,
    "bf16": (compute_dtype == torch.bfloat16 and torch.cuda.is_available()),
    "fp16": (compute_dtype == torch.float16 and torch.cuda.is_available()),
    "report_to": "none",
    "seed": SEED,
}

if "warmup_steps" in training_args_params:
    training_args_kwargs["warmup_steps"] = warmup_steps
elif "warmup_ratio" in training_args_params:
    training_args_kwargs["warmup_ratio"] = 0.03

if "evaluation_strategy" in training_args_params:
    training_args_kwargs["evaluation_strategy"] = "steps"
elif "eval_strategy" in training_args_params:
    training_args_kwargs["eval_strategy"] = "steps"

if "no_cuda" in training_args_params:
    training_args_kwargs["no_cuda"] = not torch.cuda.is_available()
elif "use_cpu" in training_args_params:
    training_args_kwargs["use_cpu"] = not torch.cuda.is_available()

unsupported_training_args = sorted(
    key for key in training_args_kwargs.keys() if key not in training_args_params
)
if unsupported_training_args:
    print(f"Skipping unsupported TrainingArguments fields: {unsupported_training_args}")

filtered_training_args_kwargs = {
    key: value for key, value in training_args_kwargs.items() if key in training_args_params
}
training_args = TrainingArguments(**filtered_training_args_kwargs)

trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())

trainer_kwargs = {
    "model": model,
    "train_dataset": train_ds,
    "eval_dataset": eval_ds,
    "args": training_args,
}

if "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer

optional_trainer_kwargs = {
    "peft_config": peft_config,
    "dataset_text_field": "text",
    "max_seq_length": MAX_SEQ_LENGTH,
    "packing": False,
}
for key, value in optional_trainer_kwargs.items():
    if value is None:
        continue
    if key in trainer_params:
        trainer_kwargs[key] = value

expected_optional = ["tokenizer", "processing_class", "peft_config", "dataset_text_field", "max_seq_length", "packing"]
unsupported_trainer_keys = sorted(key for key in expected_optional if key not in trainer_params)
if unsupported_trainer_keys:
    print(f"SFTTrainer signature does not expose: {unsupported_trainer_keys}")

trainer = SFTTrainer(**trainer_kwargs)

trainer

SFTTrainer signature does not expose: ['dataset_text_field', 'max_seq_length', 'packing', 'tokenizer']


Adding EOS to train dataset:   0%|          | 0/736 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/736 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

In [7]:
train_result = trainer.train()
metrics = train_result.metrics

trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

adapter_dir = OUTPUT_DIR / "adapter"
trainer.model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

print(f"Adapter saved to: {adapter_dir}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss
20,2.156672,2.037737
40,1.712656,1.662438
60,1.540817,1.573515
80,1.542570,1.538946
100,1.549846,1.515937
120,1.585800,1.498063
140,1.471709,1.485548
160,1.586299,1.474592
180,1.479436,1.467234
200,1.484683,1.462599


***** train metrics *****
  total_flos               = 35115415GF
  train_loss               =     1.5928
  train_runtime            = 0:49:22.39
  train_samples_per_second =      0.745
  train_steps_per_second   =      0.093
Adapter saved to: /workspaces/Apocalypse-Training/artifacts/llama31-cyber-qlora/adapter


In [10]:
import re

signals = [
    "Observed behavior linked to Wi-Fi Discovery.", "Threat intel context: Adversaries may search for information about Wi-Fi networks, such as network names and passwords, on compromised systems. Adversaries may use Wi-Fi information as part of [Account Discovery], [Remote System Discovery], and other discovery or [Credential Access] activity to support both ongoing and future campaigns. Adversaries may collect various types of information about Wi-Fi networks from hosts. For example, on Windows names and passwords of all Wi-Fi networks a device has previously connected to may be available through `netsh wlan show profiles` to enumerate Wi-Fi names and then `netsh wlan show profile Wi-Fi name key=clear` to show a Wi-Fi networks corresponding password.(Citation: BleepingComputer Agent Tesla steal wifi passwords)(Citation: Malware Bytes New AgentTesla variant steals WiFi credentials)(Citation: Check Point APT35 CharmPower January 2022) Additionally, names and other details of locally reachable Wi-Fi networks can be discovered using calls to `wlanAPI.dll` [Native API](https://attack.mitre.org/techniques/T1106) functions.(Citation: Binary Defense Emotes Wi-Fi Spreader) On Linux, names and passwords of all Wi-Fi-networks a device has previously connected to may be available in files under ` /etc/NetworkManager/system-connections/`.(Citation: Wi-Fi Password of All Connected Networks in Windows/Linux) On macOS, the password of a known Wi-Fi may be identified with ` security find-generic-password -wa wifiname` (requires admin username/password).(Citation: Find Wi-Fi Password on Mac",
]

prompt = (
    "### Instruction:\n"
    "Identify the most likely MITRE ATT&CK tactic and primary technique from observed behavior signals.\n\n"
    "### Input:\n"
    f"{json.dumps({'signals': signals}, ensure_ascii=True)}\n\n"
    "### Expected Output:\n"
)

# Switch to inference mode after training to avoid generation-time warnings.
trainer.model.eval()
if hasattr(trainer.model, "gradient_checkpointing_disable"):
    trainer.model.gradient_checkpointing_disable()
trainer.model.config.use_cache = True

if hasattr(trainer.model, "generation_config") and trainer.model.generation_config is not None:
    trainer.model.generation_config.do_sample = False
    trainer.model.generation_config.temperature = 1.0
    trainer.model.generation_config.top_p = 1.0

inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
with torch.no_grad():
    generated = trainer.model.generate(
        **inputs,
        max_new_tokens=80,
        min_new_tokens=8,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# Decode only generated continuation (not the original prompt).
new_token_ids = generated[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(new_token_ids, skip_special_tokens=True)

# Remove invisible control characters (for example zero-width joiners).
response = "".join(
    char for char in response
    if (char in "\n\t ") or (char.isprintable() and char not in "\u200b\u200c\u200d\ufeff")
).strip()

# Keep only the first JSON object to avoid trailing continuation.
json_match = re.search(r"\{[\s\S]*?\}", response)
if json_match:
    candidate = json_match.group(0)
    try:
        parsed = json.loads(candidate)
        print("Model response:\n")
        print(json.dumps(parsed, ensure_ascii=True))
    except json.JSONDecodeError:
        print("Model response:\n")
        print(candidate)
elif response:
    print("Model response:\n")
    print(response)
else:
    print("No visible text was generated. Try a second run with do_sample=True and temperature=0.7.")

Model response:

{"tactic": "discovery", "technique": "Wi-Fi Discovery"}


In [13]:
import random
import re

print("Training summary:")
print(f"- train_loss: {metrics.get('train_loss')}")
print(f"- train_runtime_seconds: {metrics.get('train_runtime')}")
print(f"- train_steps_per_second: {metrics.get('train_steps_per_second')}")
print(f"- train_samples_per_second: {metrics.get('train_samples_per_second')}")
print(f"- train_size: {len(train_ds)}")
print(f"- eval_size: {len(eval_ds)}")

def _example_expected(example):
    if isinstance(example.get("output"), dict):
        return {
            "tactic": str(example["output"].get("tactic", "")).strip(),
            "technique": str(example["output"].get("technique", "")).strip(),
        }
    return {}

def _build_prompt_from_example(example):
    signals = example.get("input", {}).get("signals", [])
    if not isinstance(signals, list):
        signals = [signals]
    return (
        "### Instruction:\n"
        "Identify the most likely MITRE ATT&CK tactic and primary technique from observed behavior signals.\n\n"
        "### Input:\n"
        f"{json.dumps({'signals': signals}, ensure_ascii=True)}\n\n"
        "### Expected Output:\n"
    )

def _generate_json(prompt_text):
    trainer.model.eval()
    if hasattr(trainer.model, "gradient_checkpointing_disable"):
        trainer.model.gradient_checkpointing_disable()
    trainer.model.config.use_cache = True

    inputs_local = tokenizer(prompt_text, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        gen_local = trainer.model.generate(
            **inputs_local,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    continuation = gen_local[0][inputs_local["input_ids"].shape[-1]:]
    text = tokenizer.decode(continuation, skip_special_tokens=True)
    text = "".join(
        ch for ch in text
        if (ch in "\n\t ") or (ch.isprintable() and ch not in "\u200b\u200c\u200d\ufeff")
    ).strip()

    match = re.search(r"\{[\s\S]*?\}", text)
    if not match:
        return None, text
    candidate_local = match.group(0)
    try:
        return json.loads(candidate_local), text
    except json.JSONDecodeError:
        return None, text

# 1) Check if the evaluated prompt resembles training data labeled as timestomp.
needle_hits = []
for idx in range(min(len(train_ds), 2000)):
    ex = train_ds[idx]
    signals = ex.get("input", {}).get("signals", [])
    if not isinstance(signals, list):
        signals = [signals]
    joined = " ".join(str(s).lower() for s in signals)
    if ("time attributes" in joined) or ("timestomp" in joined) or ("t1070.006" in joined):
        needle_hits.append((idx, _example_expected(ex), signals[:2]))
    if len(needle_hits) >= 3:
        break

print("\nExamples in train data related to time-attribute/timestomp signals:")
if not needle_hits:
    print("- none found in first 2000 rows")
else:
    for idx, expected, sigs in needle_hits:
        print(f"- row {idx}: expected={expected}")
        print(f"  signals={sigs}")

# 2) Quick sampled eval check.
sample_n = min(20, len(eval_ds))
sample_indices = list(range(sample_n)) if sample_n <= 20 else random.sample(range(len(eval_ds)), sample_n)

parsed_count = 0
exact_count = 0
tactic_count = 0
technique_count = 0
failures = []

for i in sample_indices:
    ex = eval_ds[i]
    expected = _example_expected(ex)
    prompt_text = _build_prompt_from_example(ex)
    pred, raw = _generate_json(prompt_text)

    if pred is None:
        failures.append({
            "idx": i,
            "expected": expected,
            "pred": None,
            "raw": raw[:200],
        })
        continue

    parsed_count += 1
    pred_tactic = str(pred.get("tactic", "")).strip().lower()
    pred_technique = str(pred.get("technique", "")).strip().lower()
    exp_tactic = str(expected.get("tactic", "")).strip().lower()
    exp_technique = str(expected.get("technique", "")).strip().lower()

    tactic_ok = pred_tactic == exp_tactic
    technique_ok = pred_technique == exp_technique

    if tactic_ok:
        tactic_count += 1
    if technique_ok:
        technique_count += 1
    if tactic_ok and technique_ok:
        exact_count += 1
    else:
        failures.append({
            "idx": i,
            "expected": expected,
            "pred": {
                "tactic": pred.get("tactic", ""),
                "technique": pred.get("technique", ""),
            },
            "raw": raw[:200],
        })

den = max(1, sample_n)
print("\nQuick sampled eval (deterministic decoding):")
print(f"- sample_n: {sample_n}")
print(f"- parsed_json_rate: {parsed_count}/{sample_n} = {parsed_count/den:.2f}")
print(f"- tactic_exact: {tactic_count}/{sample_n} = {tactic_count/den:.2f}")
print(f"- technique_exact: {technique_count}/{sample_n} = {technique_count/den:.2f}")
print(f"- full_exact: {exact_count}/{sample_n} = {exact_count/den:.2f}")

print("\nFirst 3 mismatches:")
for item in failures[:3]:
    print(item)

Training summary:
- train_loss: 1.5927729498648988
- train_runtime_seconds: 2962.3922
- train_steps_per_second: 0.093
- train_samples_per_second: 0.745
- train_size: 736
- eval_size: 210

Examples in train data related to time-attribute/timestomp signals:
- row 140: expected={'tactic': 'defense_evasion', 'technique': 'Timestomp'}
  signals=['Observed behavior linked to Timestomp (T1070.006).', 'Threat intel context: Adversaries may modify file time attributes to hide new files or changes to existing files. Timestomping is a technique that modifies the timestamps of a file (the modify, access, create, and change times), often to mimic files that are in the same folder and blend malicious files with legitimate files. In Windows systems, both the `$STANDARD_INFORMATION` (`$SI`) and `$FILE_NAME` (`$FN`) attributes record times in a Master File Table (MFT) file.(Citation: Inversecos Timestomping 2022) `$SI` (dates/time stamps) is displayed to the end user, including in the File System view,

In [11]:
from collections import Counter, defaultdict

def _safe_tactic(example):
    return str(example.get("output", {}).get("tactic", "")).strip()

def _safe_technique(example):
    return str(example.get("output", {}).get("technique", "")).strip()

train_tactic_counts = Counter(_safe_tactic(ex) for ex in train_ds)
eval_tactic_counts = Counter(_safe_tactic(ex) for ex in eval_ds)

print("Train tactic distribution (top 10):")
for name, count in train_tactic_counts.most_common(10):
    print(f"- {name}: {count}")

print("\nEval tactic distribution (top 10):")
for name, count in eval_tactic_counts.most_common(10):
    print(f"- {name}: {count}")

technique_to_tactics = defaultdict(set)
for ex in train_ds:
    technique = _safe_technique(ex).lower()
    tactic = _safe_tactic(ex).lower()
    if technique:
        technique_to_tactics[technique].add(tactic)

ambiguous = sorted(
    ((tech, sorted(list(tactics))) for tech, tactics in technique_to_tactics.items() if len(tactics) > 1),
    key=lambda item: len(item[1]),
    reverse=True,
)

print(f"\nTechniques mapped to multiple tactics in train set: {len(ambiguous)}")
print("First 10 ambiguous techniques:")
for tech, tactics in ambiguous[:10]:
    print(f"- {tech}: {tactics}")

Train tactic distribution (top 10):
- defense_evasion: 44
- discovery: 31
- command_and_control: 16
- credential_access: 16
- collection: 16
- execution: 16
- persistence: 15
- impact: 15
- reconnaissance: 11
- exfiltration: 9

Eval tactic distribution (top 10):
- command_and_control: 15
- credential_access: 15
- exfiltration: 15
- discovery: 15
- lateral_movement: 15
- defense_evasion: 15
- collection: 15
- execution: 15
- privilege_escalation: 15
- persistence: 15

Techniques mapped to multiple tactics in train set: 0
First 10 ambiguous techniques:


In [9]:
def _norm_label(value):
    text = str(value or "").strip().lower().replace("_", " ")
    return " ".join(text.split())

sample_n = min(20, len(eval_ds))
indices = list(range(sample_n))
norm_tactic = 0
norm_technique = 0
norm_exact = 0
parsed_count_norm = 0

for i in indices:
    ex = eval_ds[i]
    expected = _example_expected(ex)
    prompt_text = _build_prompt_from_example(ex)
    pred, _ = _generate_json(prompt_text)
    if pred is None:
        continue
    parsed_count_norm += 1
    t_ok = _norm_label(pred.get("tactic", "")) == _norm_label(expected.get("tactic", ""))
    te_ok = _norm_label(pred.get("technique", "")) == _norm_label(expected.get("technique", ""))
    if t_ok:
        norm_tactic += 1
    if te_ok:
        norm_technique += 1
    if t_ok and te_ok:
        norm_exact += 1

den = max(1, sample_n)
print("Normalized sampled eval (spaces/underscores/case-insensitive):")
print(f"- sample_n: {sample_n}")
print(f"- parsed_json_rate: {parsed_count_norm}/{sample_n} = {parsed_count_norm/den:.2f}")
print(f"- tactic_exact_norm: {norm_tactic}/{sample_n} = {norm_tactic/den:.2f}")
print(f"- technique_exact_norm: {norm_technique}/{sample_n} = {norm_technique/den:.2f}")
print(f"- full_exact_norm: {norm_exact}/{sample_n} = {norm_exact/den:.2f}")

Normalized sampled eval (spaces/underscores/case-insensitive):
- sample_n: 20
- parsed_json_rate: 20/20 = 1.00
- tactic_exact_norm: 14/20 = 0.70
- technique_exact_norm: 20/20 = 1.00
- full_exact_norm: 14/20 = 0.70


In [14]:
from pathlib import Path
import json
import shutil

checkpoint_name = "atomic-v1-stable"
target_dir = Path("/content") / checkpoint_name
zip_path = Path("/content") / f"{checkpoint_name}.zip"

source_candidates = []
if "adapter_dir" in globals():
    source_candidates.append(Path(adapter_dir))
if "OUTPUT_DIR" in globals():
    source_candidates.append(Path(OUTPUT_DIR) / "adapter")
source_candidates.append(Path("/workspaces/Apocalypse-Training/artifacts/llama31-cyber-qlora/adapter"))

source_dir = next((path for path in source_candidates if path.exists()), None)
assert source_dir is not None, f"Could not find adapter checkpoint in candidates: {source_candidates}"

if target_dir.exists():
    shutil.rmtree(target_dir)
shutil.copytree(source_dir, target_dir)

artifact_candidates = [
    Path(OUTPUT_DIR) / "train_results.json" if "OUTPUT_DIR" in globals() else Path("/tmp/nope"),
    Path(OUTPUT_DIR) / "trainer_state.json" if "OUTPUT_DIR" in globals() else Path("/tmp/nope"),
]
for artifact in artifact_candidates:
    if artifact.exists():
        shutil.copy2(artifact, target_dir / artifact.name)

manifest = {
    "checkpoint_name": checkpoint_name,
    "source_dir": str(source_dir),
    "target_dir": str(target_dir),
    "zip_path": str(zip_path),
    "train_metrics": metrics if "metrics" in globals() else {},
}
with (target_dir / "checkpoint_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, ensure_ascii=True, indent=2)

if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir="/content", base_dir=checkpoint_name)

print("Checkpoint exported successfully.")
print(f"- Folder: {target_dir}")
print(f"- Zip: {zip_path}")
print("Use Colab file browser to download the folder or the zip file.")

Checkpoint exported successfully.
- Folder: /content/atomic-v1-stable
- Zip: /content/atomic-v1-stable.zip
Use Colab file browser to download the folder or the zip file.


In [12]:
import json
import re
from collections import defaultdict
from pathlib import Path

def _normalize_label(text):
    value = str(text or "").strip().lower().replace("_", " ")
    return " ".join(value.split())

def _generate_structured_prediction(signals, max_new_tokens=96):
    prompt = (
        "### Instruction:\n"
        "Identify the most likely MITRE ATT&CK tactic and primary technique from observed behavior signals.\n\n"
        "### Input:\n"
        f"{json.dumps({'signals': signals}, ensure_ascii=True)}\n\n"
        "### Expected Output:\n"
    )

    trainer.model.eval()
    if hasattr(trainer.model, "gradient_checkpointing_disable"):
        trainer.model.gradient_checkpointing_disable()
    trainer.model.config.use_cache = True

    inputs_local = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        generated_local = trainer.model.generate(
            **inputs_local,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated_local[0][inputs_local["input_ids"].shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    raw = "".join(
        ch for ch in raw
        if (ch in "\n\t ") or (ch.isprintable() and ch not in "\u200b\u200c\u200d\ufeff")
    ).strip()

    match = re.search(r"\{[\s\S]*?\}", raw)
    if not match:
        return None, raw

    candidate = match.group(0)
    try:
        parsed = json.loads(candidate)
        return parsed, raw
    except json.JSONDecodeError:
        return None, raw

def _eval_case(case):
    pred1, raw1 = _generate_structured_prediction(case["signals"] )
    pred2, raw2 = _generate_structured_prediction(case["signals"] )

    parse_ok = pred1 is not None
    schema_ok = parse_ok and isinstance(pred1.get("tactic", None), str) and isinstance(pred1.get("technique", None), str) and bool(pred1.get("tactic", "").strip()) and bool(pred1.get("technique", "").strip())
    stable_repeat = False
    if pred1 is not None and pred2 is not None:
        stable_repeat = (
            _normalize_label(pred1.get("tactic", "")) == _normalize_label(pred2.get("tactic", ""))
            and _normalize_label(pred1.get("technique", "")) == _normalize_label(pred2.get("technique", ""))
        )

    expected_tactics = [_normalize_label(value) for value in case.get("expected_tactics", [])]
    expected_techniques = [_normalize_label(value) for value in case.get("expected_techniques", [])]

    expected_match = False
    if pred1 is not None:
        pred_tactic = _normalize_label(pred1.get("tactic", ""))
        pred_technique = _normalize_label(pred1.get("technique", ""))
        tactic_ok = (not expected_tactics) or (pred_tactic in expected_tactics)
        technique_ok = (not expected_techniques) or (pred_technique in expected_techniques)
        expected_match = tactic_ok and technique_ok

    return {
        "id": case["id"],
        "category": case["category"],
        "parse_ok": parse_ok,
        "schema_ok": schema_ok,
        "stable_repeat": stable_repeat,
        "expected_match": expected_match,
        "prediction": pred1,
        "raw_preview": (raw1[:220] if raw1 else ""),
    }

cases = [
    {
        "id": "para_wifi_1",
        "category": "paraphrased",
        "signals": [
            "Observed behavior consistent with host-based discovery of nearby wireless SSIDs and saved credentials.",
            "Threat intel context: attacker queries Wi-Fi profile data and attempts to reveal keys from local system stores.",
        ],
        "expected_tactics": ["discovery"],
        "expected_techniques": ["Wi-Fi Discovery"],
    },
    {
        "id": "para_timestomp_1",
        "category": "paraphrased",
        "signals": [
            "Observed behavior indicates manipulated file MAC times to blend payloads with older benign files.",
            "Threat intel context: attacker modifies timestamp metadata to reduce forensic visibility.",
        ],
        "expected_tactics": ["defense_evasion"],
        "expected_techniques": ["Timestomp", "Indicator Removal"],
    },
    {
        "id": "para_obfuscation_1",
        "category": "paraphrased",
        "signals": [
            "Observed behavior suggests C2 traffic intentionally disguised to hinder straightforward inspection.",
            "Threat intel context: extra junk bytes and protocol shaping were used to conceal command content.",
        ],
        "expected_tactics": ["command_and_control"],
        "expected_techniques": ["Data Obfuscation", "Junk Data", "Protocol or Service Impersonation"],
    },
    {
        "id": "noise_wifi_1",
        "category": "noisy_logs",
        "signals": [
            "INFO backup job completed successfully for /var/archive",
            "Observed behavior linked to Wi-Fi Discovery (T1016.002).",
            "DEBUG scheduled telemetry heartbeat to internal metrics endpoint",
            "Threat intel context: attacker enumerates wireless profiles and retrieves network key material.",
        ],
        "expected_tactics": ["discovery"],
        "expected_techniques": ["Wi-Fi Discovery"],
    },
    {
        "id": "noise_rdp_1",
        "category": "noisy_logs",
        "signals": [
            "INFO user opened spreadsheet and printed report",
            "Observed behavior linked to Remote Desktop Protocol (T1021.001).",
            "WARN packet loss detected on non-critical monitoring network",
            "Threat intel context: remote interactive session established across internal hosts.",
        ],
        "expected_tactics": ["lateral_movement"],
        "expected_techniques": ["Remote Desktop Protocol"],
    },
    {
        "id": "noise_ssh_1",
        "category": "noisy_logs",
        "signals": [
            "INFO package update completed: openssl",
            "Observed behavior linked to SSH (T1021.004).",
            "DEBUG cron executed housekeeping script",
            "Threat intel context: remote command channel over SSH used to reach additional systems.",
        ],
        "expected_tactics": ["lateral_movement"],
        "expected_techniques": ["SSH"],
    },
    {
        "id": "adv_wifi_typo_1",
        "category": "adversarial_like",
        "signals": [
            "Observed behavior linked to W1-F1 Discovry (T1016.002) with slight telemetry corruption.",
            "Threat intel context: command output still shows enumeration of nearby wireless network identifiers and saved keys.",
        ],
        "expected_tactics": ["discovery"],
        "expected_techniques": ["Wi-Fi Discovery"],
    },
    {
        "id": "adv_timestomp_hedge_1",
        "category": "adversarial_like",
        "signals": [
            "Observed behavior linked to Timestomp (T1070.006), although process comment claims normal file maintenance.",
            "Threat intel context: multiple files had backdated modify/access/create timestamps shortly after payload execution.",
        ],
        "expected_tactics": ["defense_evasion"],
        "expected_techniques": ["Timestomp", "Indicator Removal"],
    },
    {
        "id": "adv_obf_mixed_1",
        "category": "adversarial_like",
        "signals": [
            "Observed behavior linked to Data Obfuscation (T1001) with occasional fake TLS handshake markers.",
            "Threat intel context: payload stream includes meaningless inserted bytes and protocol impersonation artifacts.",
        ],
        "expected_tactics": ["command_and_control"],
        "expected_techniques": ["Data Obfuscation", "Junk Data", "Protocol or Service Impersonation"],
    },
    {
        "id": "border_scan_vs_vuln_1",
        "category": "borderline",
        "signals": [
            "Observed behavior shows high-rate address sweeping across multiple /24 ranges and follow-on CVE probe requests.",
            "Threat intel context: campaign appears between broad scanning and vulnerability-focused enumeration.",
        ],
        "expected_tactics": ["reconnaissance"],
        "expected_techniques": ["Scanning IP Blocks", "Vulnerability Scanning", "Active Scanning"],
    },
    {
        "id": "border_rdp_vs_ssh_1",
        "category": "borderline",
        "signals": [
            "Observed behavior indicates remote admin attempts over both 3389 and 22 during host pivot.",
            "Threat intel context: attacker alternates protocol choice based on reachable service exposure.",
        ],
        "expected_tactics": ["lateral_movement"],
        "expected_techniques": ["Remote Desktop Protocol", "SSH"],
    },
    {
        "id": "border_obf_vs_impersonation_1",
        "category": "borderline",
        "signals": [
            "Observed behavior includes protocol traffic shaped to resemble legitimate web services while embedding junk bytes.",
            "Threat intel context: C2 pattern lies between general obfuscation and explicit protocol/service impersonation.",
        ],
        "expected_tactics": ["command_and_control"],
        "expected_techniques": ["Data Obfuscation", "Protocol or Service Impersonation", "Junk Data"],
    },
]

results = [_eval_case(case) for case in cases]
summary = {
    "total_cases": len(results),
    "parse_rate": sum(1 for r in results if r["parse_ok"]) / max(1, len(results)),
    "schema_rate": sum(1 for r in results if r["schema_ok"]) / max(1, len(results)),
    "repeat_stability_rate": sum(1 for r in results if r["stable_repeat"]) / max(1, len(results)),
    "expected_match_rate": sum(1 for r in results if r["expected_match"]) / max(1, len(results)),
}

by_category = defaultdict(list)
for row in results:
    by_category[row["category"]].append(row)

category_metrics = {}
for category, rows in by_category.items():
    total = len(rows)
    category_metrics[category] = {
        "cases": total,
        "parse_rate": sum(1 for r in rows if r["parse_ok"]) / max(1, total),
        "schema_rate": sum(1 for r in rows if r["schema_ok"]) / max(1, total),
        "repeat_stability_rate": sum(1 for r in rows if r["stable_repeat"]) / max(1, total),
        "expected_match_rate": sum(1 for r in rows if r["expected_match"]) / max(1, total),
    }

report = {
    "checkpoint": "/content/atomic-v1-stable",
    "focus": "stability_of_representation",
    "summary": summary,
    "category_metrics": category_metrics,
    "results": results,
}

report_path = Path("/content/atomic-v1-stable") / "evaluation_stability_report.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
with report_path.open("w", encoding="utf-8") as handle:
    json.dump(report, handle, ensure_ascii=True, indent=2)

print("Stability evaluation completed.")
print("Summary:")
print(json.dumps(summary, ensure_ascii=True, indent=2))
print("\nBy category:")
print(json.dumps(category_metrics, ensure_ascii=True, indent=2))
print(f"\nReport saved to: {report_path}")

Stability evaluation completed.
Summary:
{
  "total_cases": 12,
  "parse_rate": 1.0,
  "schema_rate": 1.0,
  "repeat_stability_rate": 1.0,
  "expected_match_rate": 0.16666666666666666
}

By category:
{
  "paraphrased": {
    "cases": 3,
    "parse_rate": 1.0,
    "schema_rate": 1.0,
    "repeat_stability_rate": 1.0,
    "expected_match_rate": 0.0
  },
  "noisy_logs": {
    "cases": 3,
    "parse_rate": 1.0,
    "schema_rate": 1.0,
    "repeat_stability_rate": 1.0,
    "expected_match_rate": 0.3333333333333333
  },
  "adversarial_like": {
    "cases": 3,
    "parse_rate": 1.0,
    "schema_rate": 1.0,
    "repeat_stability_rate": 1.0,
    "expected_match_rate": 0.3333333333333333
  },
  "borderline": {
    "cases": 3,
    "parse_rate": 1.0,
    "schema_rate": 1.0,
    "repeat_stability_rate": 1.0,
    "expected_match_rate": 0.0
  }
}

Report saved to: /content/atomic-v1-stable/evaluation_stability_report.json
